<a href="https://colab.research.google.com/github/kanat-analytics/youtube-data-cleaning/blob/main/youtube_api.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [43]:
from google.colab import userdata

api_key = userdata.get('YOUTUBE_API_KEY')

print('Key loaded:', len(api_key) > 0)
print('Key length:', len(api_key))

Key loaded: True
Key length: 39


In [44]:
import requests

url = 'https://www.googleapis.com/youtube/v3/videos'
params = {
    'part': 'snippet,statistics',
    'id': 'Wk-RTs0r8EA',
    'key': api_key
}

response = requests.get(url, params=params)
print('Status:', response.status_code)

Status: 200


In [45]:
import json

data = response.json()
print(json.dumps(data, indent=2))

{
  "kind": "youtube#videoListResponse",
  "etag": "jvEJJ6KsyHrO4_5OX1VOXqM9n7A",
  "items": [
    {
      "kind": "youtube#video",
      "etag": "m9Ez34FIYWuw_a1x-Oppq8FgPb0",
      "id": "Wk-RTs0r8EA",
      "snippet": {
        "publishedAt": "2026-10-06T12:03:17Z",
        "channelId": "UCINGv5BTWX46SKotKLWPn4Q",
        "title": "Ocean waves",
        "description": "Video: Pexels (free license)",
        "thumbnails": {
          "default": {
            "url": "https://i.ytimg.com/vi/Wk-RTs0r8EA/default.jpg",
            "width": 120,
            "height": 90
          },
          "medium": {
            "url": "https://i.ytimg.com/vi/Wk-RTs0r8EA/mqdefault.jpg",
            "width": 320,
            "height": 180
          },
          "high": {
            "url": "https://i.ytimg.com/vi/Wk-RTs0r8EA/hqdefault.jpg",
            "width": 480,
            "height": 360
          },
          "standard": {
            "url": "https://i.ytimg.com/vi/Wk-RTs0r8EA/sddefault.jpg",
     

In [46]:
print(data.keys())

dict_keys(['kind', 'etag', 'items', 'pageInfo'])


In [47]:
items = data['items']
print(type(items))
print(len(items))

<class 'list'>
1


In [48]:
video = items[0]
print(video.keys())

dict_keys(['kind', 'etag', 'id', 'snippet', 'statistics'])


In [49]:
print(video['id'])
print(video['snippet']['title'])
print(video['statistics']['viewCount'])

Wk-RTs0r8EA
Ocean waves
2


In [50]:
from IPython.core.display import Video
print(video['statistics']['likeCount'])
print(video['statistics']['viewCount'])

0
2


In [51]:
import pandas as pd

row = {
    'video_id': video['id'],
    'title': video['snippet']['title'],
    'published_at': video['snippet']['publishedAt'],
    'views': video['statistics']['viewCount'],
    'likes': video['statistics']['likeCount'],
    'comments': video['statistics']['commentCount']
}

df_api = pd.DataFrame([row])
df_api

,video_id,title,published_at,views,likes,comments
0,Wk-RTs0r8EA,Ocean waves,2026-10-06T12:03:17Z,2,0,0


In [52]:
print(df_api.dtypes)

video_id        object
title           object
published_at    object
views           object
likes           object
comments        object
dtype: object


In [53]:
cols = ['views', 'likes', 'comments']
df_api[cols] = df_api[cols].astype(int)

df_api['published_at'] = pd.to_datetime(df_api['published_at'])

print(df_api.dtypes)

video_id                     object
title                        object
published_at    datetime64[ns, UTC]
views                         int64
likes                         int64
comments                      int64
dtype: object


In [54]:
channel_id = video['snippet']['channelId']
print(channel_id)

UCINGv5BTWX46SKotKLWPn4Q


In [55]:
url_ch = 'https://www.googleapis.com/youtube/v3/channels'
params_ch = {
    'part': 'snippet,statistics,contentDetails',
    'id': channel_id,
    'key': api_key
}

response_ch = requests.get(url_ch, params=params_ch)
print('Status:', response_ch.status_code)

channel = response_ch.json()['items'][0]
print(channel['snippet']['title'])
print(channel['statistics'])

Status: 200
Kanat Kaiykhov
{'viewCount': '3', 'subscriberCount': '0', 'hiddenSubscriberCount': False, 'videoCount': '2'}


In [56]:
uploads_id = channel['contentDetails']['relatedPlaylists']['uploads']
print(uploads_id)

UUINGv5BTWX46SKotKLWPn4Q


In [57]:
url_pl = 'https://www.googleapis.com/youtube/v3/playlistItems'
params_pl = {
    'part': 'contentDetails',
    'playlistId': uploads_id,
    'maxResults': 50,
    'key': api_key
}

response_pl = requests.get(url_pl, params=params_pl)
print('Status:', response_pl.status_code)

playlist = response_pl.json()
print('Videos:', len(playlist['items']))
print(playlist['items'][0]['contentDetails'])

Status: 200
Videos: 2
{'videoId': 'UyD6Wfu5wh0', 'videoPublishedAt': '2026-10-06T12:06:15Z'}


In [58]:
print(playlist['items'][0]['contentDetails']['videoId'])
print(playlist['items'][1]['contentDetails']['videoId'])

UyD6Wfu5wh0
Wk-RTs0r8EA


In [59]:
for item in playlist['items']:
    print(item['contentDetails']['videoId'])

UyD6Wfu5wh0
Wk-RTs0r8EA


In [60]:
video_ids = []
for item in playlist['items']:
    video_ids.append(item['contentDetails']['videoId'])

print(video_ids)
print('Total:', len(video_ids))

['UyD6Wfu5wh0', 'Wk-RTs0r8EA']
Total: 2


In [61]:
ids_text = ','.join(video_ids)
print(ids_text)

UyD6Wfu5wh0,Wk-RTs0r8EA


In [62]:
url = 'https://www.googleapis.com/youtube/v3/videos'
params = {
    'part': 'snippet,statistics',
    'id': ids_text,
    'key': api_key
}

response = requests.get(url, params=params)
print('Status:', response.status_code)

data = response.json()
print('Videos:', len(data['items']))

Status: 200
Videos: 2


In [63]:
import pandas as pd

rows = []
for video in data['items']:
    row = {
        'video_id': video['id'],
        'title': video['snippet']['title'],
        'published_at': video['snippet']['publishedAt'],
        'views': video['statistics']['viewCount'],
        'likes': video['statistics']['likeCount'],
        'comments': video['statistics']['commentCount']
    }
    rows.append(row)

df_videos = pd.DataFrame(rows)
df_videos

,video_id,title,published_at,views,likes,comments
0,UyD6Wfu5wh0,Ocean from above,2026-10-06T12:06:15Z,1,0,0
1,Wk-RTs0r8EA,Ocean waves,2026-10-06T12:03:17Z,2,0,0


In [64]:
test_stats = {'viewCount': '500', 'commentCount': '3'}

print(test_stats.get('viewCount'))
print(test_stats.get('likeCount'))

500
None


In [65]:
rows = []
for video in data['items']:
    stats = video['statistics']
    row = {
        'video_id': video['id'],
        'title': video['snippet']['title'],
        'published_at': video['snippet']['publishedAt'],
        'views': stats.get('viewCount'),
        'likes': stats.get('likeCount'),
        'comments': stats.get('commentCount')
    }
    rows.append(row)

df_videos = pd.DataFrame(rows)
df_videos

,video_id,title,published_at,views,likes,comments
0,UyD6Wfu5wh0,Ocean from above,2026-10-06T12:06:15Z,1,0,0
1,Wk-RTs0r8EA,Ocean waves,2026-10-06T12:03:17Z,2,0,0


In [66]:
cols = ['views', 'likes', 'comments']
for col in cols:
    df_videos[col] = pd.to_numeric(df_videos[col])

df_videos['published_at'] = pd.to_datetime(df_videos['published_at'])

print(df_videos.dtypes)

video_id                     object
title                        object
published_at    datetime64[ns, UTC]
views                         int64
likes                         int64
comments                      int64
dtype: object


In [67]:
print('Rows:', len(df_videos))
print('Expected videos:', len(video_ids))
print('Duplicate video_id:', df_videos['video_id'].duplicated().sum())
print('Invalid video_id length:', (df_videos['video_id'].str.len() != 11).sum())
print('Missing values:', df_videos[cols].isnull().sum().sum())
print('Negative values:', (df_videos[cols] < 0).sum().sum())
print('Total views:', df_videos['views'].sum())

Rows: 2
Expected videos: 2
Duplicate video_id: 0
Invalid video_id length: 0
Missing values: 0
Negative values: 0
Total views: 3


In [68]:
url_pl = 'https://www.googleapis.com/youtube/v3/playlistItems'
params_pl = {
    'part': 'contentDetails',
    'playlistId': uploads_id,
    'maxResults': 1,
    'key': api_key
}

response_pl = requests.get(url_pl, params=params_pl)
page1 = response_pl.json()

print('Videos on page 1:', len(page1['items']))
print('Next page token:', page1.get('nextPageToken'))
print('Total results:', page1['pageInfo']['totalResults'])

Videos on page 1: 1
Next page token: EAAaHlBUOkNBRWlFREF4TnpJd09FWkJRVGcxTWpNelJqaw
Total results: 2


In [69]:
params_pl['pageToken'] = page1['nextPageToken']

response_pl = requests.get(url_pl, params=params_pl)
page2 = response_pl.json()

print('Videos on page 2:', len(page2['items']))
print('Next page token:', page2.get('nextPageToken'))

Videos on page 2: 1
Next page token: None


In [70]:
video_ids = []
next_token = None
page_number = 0

while True:
    params_pl = {
        'part': 'contentDetails',
        'playlistId': uploads_id,
        'maxResults': 1,
        'key': api_key
    }
    if next_token:
        params_pl['pageToken'] = next_token

    response_pl = requests.get(url_pl, params=params_pl)
    page = response_pl.json()

    for item in page['items']:
        video_ids.append(item['contentDetails']['videoId'])

    page_number = page_number + 1
    print('Page', page_number, '-', len(page['items']), 'videos')

    next_token = page.get('nextPageToken')
    if next_token is None:
        break

print(video_ids)
print('Total:', len(video_ids))

Page 1 - 1 videos
Page 2 - 1 videos
['UyD6Wfu5wh0', 'Wk-RTs0r8EA']
Total: 2


In [71]:
ids_text = ','.join(video_ids)

url = 'https://www.googleapis.com/youtube/v3/videos'
params = {
    'part': 'contentDetails',
    'id': ids_text,
    'key': api_key
}

response = requests.get(url, params=params)
data_dur = response.json()

for video in data_dur['items']:
    print(video['id'], video['contentDetails']['duration'])

UyD6Wfu5wh0 PT41S
Wk-RTs0r8EA PT8S


In [72]:
durations = []
for video in data_dur['items']:
    durations.append({
        'video_id': video['id'],
        'duration_raw': video['contentDetails']['duration']
    })

df_dur = pd.DataFrame(durations)
df_dur['duration_sec'] = pd.to_timedelta(df_dur['duration_raw']).dt.total_seconds().astype(int)
df_dur['duration_min'] = (df_dur['duration_sec'] / 60).round(2)
df_dur

,video_id,duration_raw,duration_sec,duration_min
0,UyD6Wfu5wh0,PT41S,41,0.68
1,Wk-RTs0r8EA,PT8S,8,0.13


In [73]:
df_videos = df_videos.merge(df_dur[['video_id', 'duration_sec', 'duration_min']],
                            on='video_id', how='left')
df_videos

,video_id,title,published_at,views,likes,comments,duration_sec,duration_min
0,UyD6Wfu5wh0,Ocean from above,2026-10-06 12:06:15+00:00,1,0,0,41,0.68
1,Wk-RTs0r8EA,Ocean waves,2026-10-06 12:03:17+00:00,2,0,0,8,0.13


In [74]:
url = 'https://www.googleapis.com/youtube/v3/videos'
params = {
    'part': 'snippet,statistics',
    'id': 'Wk-RTs0r8EA',
    'key': 'wrong_key'
}

response = requests.get(url, params=params)

if response.status_code == 200:
    data = response.json()
    print('OK, videos:', len(data['items']))
else:
    print('Error code:', response.status_code)
    print('Message:', response.json()['error']['message'])

Error code: 400
Message: API key not valid. Please pass a valid API key.


In [75]:
    'key': api_key

SyntaxError: illegal target for annotation (1976520568.py, line 1)

In [ ]:
url = 'https://www.googleapis.com/youtube/v3/videos'
params = {
    'part': 'snippet,statistics',
    'id': 'Wk-RTs0r8EA',
    'key': api_key
}

response = requests.get(url, params=params)

if response.status_code == 200:
    data = response.json()
    print('OK, videos:', len(data['items']))
else:
    print('Error code:', response.status_code)
    print('Message:', response.json()['error']['message'])

In [ ]:
try:
    response = requests.get('https://www.youtube-api-wrong-address.com', timeout=10)
    print('Status:', response.status_code)
except Exception as e:
    print('Request failed:', type(e).__name__)

In [ ]:
url = 'https://www.googleapis.com/youtube/v3/videos'
params = {
    'part': 'snippet,statistics',
    'id': 'Wk-RTs0r8EA',
    'key': api_key
}

try:
    response = requests.get(url, params=params, timeout=10)
    if response.status_code == 200:
        data = response.json()
        print('OK, videos:', len(data['items']))
    else:
        print('Error code:', response.status_code)
        print('Message:', response.json()['error']['message'])
except Exception as e:
    print('Request failed:', type(e).__name__)

In [ ]:
import math

videos_count = 300
pages = math.ceil(videos_count / 50)
units = 1 + pages + pages

print('Pages:', pages)
print('Units per run:', units)
print('Units left today:', 10000 - units)

In [ ]:
import json
from google.colab import userdata

client_config = json.loads(userdata.get('YOUTUBE_OAUTH_CLIENT'))

print('Keys:', list(client_config.keys()))
print('Has client_id:', 'client_id' in client_config['installed'])

In [76]:
from google_auth_oauthlib.flow import Flow

SCOPES = [
    'https://www.googleapis.com/auth/yt-analytics.readonly',
    'https://www.googleapis.com/auth/youtube.readonly'
]

flow = Flow.from_client_config(client_config, scopes=SCOPES,
                               redirect_uri='http://localhost')

auth_url, _ = flow.authorization_url(prompt='consent', access_type='offline')
print(auth_url)

https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=200498455862-st4n6h0s44llq8a9s79r7sqo81pccbha.apps.googleusercontent.com&redirect_uri=http%3A%2F%2Flocalhost&scope=https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fyt-analytics.readonly+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fyoutube.readonly&state=DCvKURkLFyg4Vv29f2NKLWzdJ3Gt7v&code_challenge=95a_fnplR_Ex_Ks97wu5GS5KJ4dno7A-vkS8ZFYmlaI&code_challenge_method=S256&prompt=consent&access_type=offline


In [ ]:

from urllib.parse import urlparse, parse_qs

redirect_url = input('Paste the full address: ')
code = parse_qs(urlparse(redirect_url).query)['code'][0]

flow.fetch_token(code=code)
creds = flow.credentials

print('Token received:', creds.valid)
print('Refresh token:', creds.refresh_token is not None)

In [80]:
print('OAuth 2.0 access granted:', creds.valid)
print('Refresh token received:', creds.refresh_token is not None)

OAuth 2.0 access granted: True
Refresh token received: True


In [78]:
import requests

url = 'https://youtubeanalytics.googleapis.com/v2/reports'
params = {
    'ids': 'channel==MINE',
    'startDate': '2026-10-01',
    'endDate': '2026-10-18',
    'metrics': 'views,estimatedMinutesWatched,averageViewDuration',
    'dimensions': 'day'
}
headers = {'Authorization': 'Bearer ' + creds.token}

response = requests.get(url, params=params, headers=headers)
print('Status:', response.status_code)

data_an = response.json()
print('Keys:', list(data_an.keys()))

Status: 200
Keys: ['kind', 'columnHeaders', 'rows']


In [79]:
import pandas as pd

columns = []
for col in data_an['columnHeaders']:
    columns.append(col['name'])

df_an = pd.DataFrame(data_an['rows'], columns=columns)
df_an

,day,views,estimatedMinutesWatched,averageViewDuration
0,2026-10-07,3,0,8
